In [1]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/raw/srodmiescie_raw.csv", sep=";", index_col=0, dtype=column_types
).dropna(subset="ground_floor_function")

data_gdynia.head()

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking
id,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,Hotel,NaN,Hotel,7,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026
2,890729731,Świętego Piotra,6D,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026
3,890292040,Węglowa,22,NaN,Centrum Handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026
4,890729732,Świętego Piotra,6,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026
5,136885205,Tadeusza Wendy,15,NaN,Centrum Medyczne,NaN,Centrum Medyczne,6,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,2,27.09.2026


In [3]:
data_extract = (
    data_gdynia.assign(
        funkcja_parter=lambda d: (
            d["ground_floor_function"]
            .str.replace("_", " ")
            .str.lower()
            .str.split(pat=",")
        )
    )
    .explode("ground_floor_function")
    .assign(funkcja_parter=lambda d: d["ground_floor_function"].str.strip())
)
data_extract

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,funkcja_parter
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,Hotel,NaN,Hotel,7,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026,Hotel
2,890729731,Świętego Piotra,6D,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026,Mieszkaniowa Wielorodzinna
3,890292040,Węglowa,22,NaN,Centrum Handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026,Centrum Handlowe
4,890729732,Świętego Piotra,6,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3,27.09.2026,Mieszkaniowa Wielorodzinna
5,136885205,Tadeusza Wendy,15,NaN,Centrum Medyczne,NaN,Centrum Medyczne,6,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,2,27.09.2026,Centrum Medyczne
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237,126180359,Władysława IV,61,NaN,"Laboratorium Medyczne,Piekarnia,Kiosk",NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1940,Modernizm,3,NaN,NaN,google.com/maps,2,30.09.2026,"Laboratorium Medyczne,Piekarnia,Kiosk"
1238,126173503,Świętojańska,122,NaN,"Apteka,Handel Zdrowie",NaN,Biurowa,6,<NA>,1930,Modernizm,2,NaN,NaN,google.com/maps,2,30.09.2026,"Apteka,Handel Zdrowie"
1239,260676369,Aleja Marszałka Józefa Piłsudskiego,56,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1930,Modernizm,2,NaN,NaN,google.com/maps,2,30.09.2026,Mieszkaniowa Wielorodzinna


In [4]:
assert data_extract["ground_floor_function"].isna().sum() == 0

In [5]:
number_functions = data_extract["ground_floor_function"].value_counts()

In [6]:
category_map_skeleton = (
    number_functions.reset_index()
    .rename(columns={"ground_floor_function": "functions"})
    .assign(function_clean="", category="", subcategory="")
)

# category_map_skeleton.to_csv(
#    "../data/reference/category_map2.csv", sep=";", index=False, encoding="utf-8"
# )